# Multi-Label Legal-BERT Classifier

**Purpose**: Fine-tune Legal-BERT for multi-label classification across all 41 CUAD
legal clause categories using the pre-tokenized dataset.

**Model**: `nlpaueb/legal-bert-base-uncased` with sigmoid + BCEWithLogitsLoss

**Input**: Pre-tokenized HuggingFace Datasets (from `tokenized_multi_label_dataset.ipynb`)

---
## Section 1 — Imports & Configuration

In [40]:
import json
import os
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report,
)

# ==========================================
# CONFIGURATION
# ==========================================

RANDOM_SEED = 42
NUM_LABELS = 41
MAX_LENGTH = 256
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
THRESHOLD = 0.3  # sigmoid probability threshold for binary prediction

# Paths
BASE_DIR = "/kaggle/input/"
TOKENIZED_DIR = os.path.join(BASE_DIR, "datasets", "charishmaganta","multi-label-dataset", "processed", "tokenized_multi_label_dataset")
TOKENIZER_DIR = os.path.join(BASE_DIR, "models","charishmaganta", "models", "other", "default", "1", "models", "legal_bert_multilabel", "tokenizer")
OUTPUT_DIR = "/kaggle/working/legal_bert_multilabel"
LABEL_MAP_PATH = os.path.join(BASE_DIR, "datasets", "charishmaganta","multi-label-dataset", "processed","label_mapping.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Configuration ready.")
print(f"  Model:       {MODEL_NAME}")
print(f"  Num labels:  {NUM_LABELS}")
print(f"  Max length:  {MAX_LENGTH}")
print(f"  Threshold:   {THRESHOLD}")
print(f"  Output dir:  {OUTPUT_DIR}")

Configuration ready.
  Model:       nlpaueb/legal-bert-base-uncased
  Num labels:  41
  Max length:  256
  Threshold:   0.3
  Output dir:  /kaggle/working/legal_bert_multilabel


---
## Section 2 — GPU / Device Detection

Training transformers on CPU is extremely slow. GPU (CUDA) is strongly recommended.
This section automatically detects available hardware.

In [41]:
# ==========================================
# DEVICE DETECTION
# ==========================================

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA version:    {torch.version.cuda}")
    print(f"  GPU memory:      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"  Device:          {device}")
    USE_FP16 = True
else:
    device = torch.device("cpu")
    print("⚠ No GPU detected — using CPU (training will be very slow)")
    print(f"  Device: {device}")
    USE_FP16 = False

print(f"  FP16 training:   {USE_FP16}")

✓ GPU detected: Tesla T4
  CUDA version:    12.8
  GPU memory:      14.6 GB
  Device:          cuda
  FP16 training:   True


---
## Section 3 — Load Tokenized Datasets

We load the pre-tokenized datasets created in `tokenized_multi_label_dataset.ipynb`.
These contain `input_ids`, `attention_mask`, `labels`, and `contract_id`.

In [42]:
# ==========================================
# LOAD TOKENIZED DATASETS
# ==========================================

train_dataset = load_from_disk(os.path.join(TOKENIZED_DIR, "train"))
test_dataset = load_from_disk(os.path.join(TOKENIZED_DIR, "test"))

print("Tokenized datasets loaded.")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")
print(f"  Features: {list(train_dataset.features.keys())}")

# Inspect a sample
sample = train_dataset[0]
print(f"\nSample structure:")
print(f"  input_ids length:      {len(sample['input_ids'])}")
print(f"  attention_mask length: {len(sample['attention_mask'])}")
print(f"  labels length:         {len(sample['labels'])}")
print(f"  labels dtype:          {type(sample['labels'][0]).__name__}")
print(f"  contract_id:           {sample['contract_id']}")

Tokenized datasets loaded.
  Train: 16073 samples
  Test:  4031 samples
  Features: ['input_ids', 'attention_mask', 'labels', 'contract_id']

Sample structure:
  input_ids length:      256
  attention_mask length: 256
  labels length:         41
  labels dtype:          float
  contract_id:           LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT


---
## Section 4 — Load Label Mapping

In [43]:
# ==========================================
# LOAD LABEL MAPPING
# ==========================================

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

label_columns = label_mapping["all_labels"]

print(f"Label mapping loaded.")
print(f"  Total labels: {len(label_columns)}")
print(f"\nAll {len(label_columns)} labels:")
for i, label in enumerate(label_columns):
    print(f"  {i:2d}. {label}")

Label mapping loaded.
  Total labels: 41

All 41 labels:
   0. Affiliate License-Licensee
   1. Affiliate License-Licensor
   2. Agreement Date
   3. Anti-Assignment
   4. Audit Rights
   5. Cap On Liability
   6. Change Of Control
   7. Competitive Restriction Exception
   8. Covenant Not To Sue
   9. Document Name
  10. Effective Date
  11. Exclusivity
  12. Expiration Date
  13. Governing Law
  14. Insurance
  15. Ip Ownership Assignment
  16. Irrevocable Or Perpetual License
  17. Joint Ip Ownership
  18. License Grant
  19. Liquidated Damages
  20. Minimum Commitment
  21. Most Favored Nation
  22. No-Solicit Of Customers
  23. No-Solicit Of Employees
  24. Non-Compete
  25. Non-Disparagement
  26. Non-Transferable License
  27. Notice Period To Terminate Renewal
  28. Parties
  29. Post-Termination Services
  30. Price Restrictions
  31. Renewal Term
  32. Revenue/Profit Sharing
  33. Rofr/Rofo/Rofn
  34. Source Code Escrow
  35. Termination For Convenience
  36. Third Party Bene

---
## Section 5 — Load Tokenizer

In [44]:
# ==========================================
# LOAD TOKENIZER
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)

print("Tokenizer loaded successfully.")
print(f"  Source:     {TOKENIZER_DIR}")
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  Max length: {tokenizer.model_max_length}")

Tokenizer loaded successfully.
  Source:     /kaggle/input/models/charishmaganta/models/other/default/1/models/legal_bert_multilabel/tokenizer
  Vocab size: 30522
  Max length: 512


---
## Section 6 — Prepare Dataset Format

The HuggingFace `Trainer` expects specific column names. We need to ensure
`input_ids`, `attention_mask`, and `labels` are present. The `contract_id`
column should be removed before training since it's not a model input.

We also set the format to PyTorch tensors.

In [45]:
# ==========================================
# PREPARE DATASET FORMAT FOR TRAINER
# ==========================================

# Remove contract_id — not a model input, only used for split verification
train_dataset_clean = train_dataset.remove_columns(["contract_id"])
test_dataset_clean = test_dataset.remove_columns(["contract_id"])

# Set format to PyTorch tensors
train_dataset_clean.set_format("torch")
test_dataset_clean.set_format("torch")

print("Datasets prepared for Trainer.")
print(f"  Train columns: {train_dataset_clean.column_names}")
print(f"  Test columns:  {test_dataset_clean.column_names}")

# Verify tensor shapes
sample = train_dataset_clean[0]
print(f"\nTensor shapes:")
print(f"  input_ids:      {sample['input_ids'].shape}")
print(f"  attention_mask: {sample['attention_mask'].shape}")
print(f"  labels:         {sample['labels'].shape}")
print(f"  labels dtype:   {sample['labels'].dtype}")

Datasets prepared for Trainer.
  Train columns: ['input_ids', 'attention_mask', 'labels']
  Test columns:  ['input_ids', 'attention_mask', 'labels']

Tensor shapes:
  input_ids:      torch.Size([256])
  attention_mask: torch.Size([256])
  labels:         torch.Size([41])
  labels dtype:   torch.float32


---
## Section 7 — Multi-Label Classification Theory

### Multi-Label vs Multi-Class

| Aspect | Multi-Class | Multi-Label |
|--------|------------|-------------|
| Labels per sample | Exactly 1 | 0 or more |
| Output activation | Softmax | Sigmoid |
| Loss function | CrossEntropyLoss | BCEWithLogitsLoss |
| Probabilities | Sum to 1.0 | Independent per label |
| Example | "This is a cat" | "This has: liability, termination, renewal" |

### What are Logits?

**Logits** are the raw, unnormalized scores output by the model's final linear layer.
They can be any real number (positive or negative). They are NOT probabilities yet.

```
Model output (logits): [-2.1, 3.4, 0.1, -0.8, 1.2, ...]
```

### What is Sigmoid?

**Sigmoid** squashes each logit independently into the range [0, 1]:

```
sigmoid(x) = 1 / (1 + exp(-x))

sigmoid(-2.1) = 0.11  → low probability
sigmoid(3.4)  = 0.97  → high probability
sigmoid(0.1)  = 0.52  → borderline
```

Unlike softmax, sigmoid treats each label **independently**. Multiple labels
can have high probability simultaneously — which is exactly what we need
for multi-label classification.

### Why NOT Softmax?

**Softmax** forces all probabilities to sum to 1.0, creating competition
between labels. If one label goes up, others must go down. This is wrong
for multi-label because a clause can be relevant to multiple categories
simultaneously (e.g., both "Cap On Liability" and "Uncapped Liability").

### What is BCEWithLogitsLoss?

**BCEWithLogitsLoss** = Binary Cross-Entropy with Logits. It:
1. Applies sigmoid internally (numerically stable)
2. Computes binary cross-entropy loss per label independently
3. Averages across all 41 labels

This is the correct loss for multi-label classification.
`CrossEntropyLoss` would be WRONG because it assumes mutually exclusive classes.

### Thresholding

After sigmoid, we get probabilities per label. To make binary predictions,
we apply a **threshold** (default 0.5):

```
probability >= 0.5  →  predict 1 (clause present)
probability <  0.5  →  predict 0 (clause absent)
```

The threshold can be tuned per label for optimal performance.

---
## Section 8 — Load Legal-BERT Model

We load Legal-BERT with a classification head configured for 41 labels
and `problem_type="multi_label_classification"` which tells HuggingFace
to use `BCEWithLogitsLoss` internally.

In [46]:
# ==========================================
# LOAD LEGAL-BERT FOR MULTI-LABEL CLASSIFICATION
# ==========================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

model.to(device)

print("Legal-BERT loaded successfully.")
print(f"  Model:         {MODEL_NAME}")
print(f"  Num labels:    {NUM_LABELS}")
print(f"  Problem type:  multi_label_classification")
print(f"  Loss function: BCEWithLogitsLoss (automatic)")
print(f"  Device:        {device}")
print(f"  Parameters:    {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable:     {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

Legal-BERT loaded successfully.
  Model:         nlpaueb/legal-bert-base-uncased
  Num labels:    41
  Problem type:  multi_label_classification
  Loss function: BCEWithLogitsLoss (automatic)
  Device:        cuda
  Parameters:    109,513,769
  Trainable:     109,513,769


---
## Section 9 — Metrics Function

This function is called by the Trainer after each evaluation.
It converts raw logits to predictions using sigmoid + threshold,
then computes all multi-label metrics.

In [47]:
# ==========================================
# COMPUTE METRICS FUNCTION
# ==========================================

def compute_metrics(eval_pred):
    """Compute multi-label classification metrics.

    Called automatically by HuggingFace Trainer during evaluation.

    Args:
        eval_pred: EvalPrediction with (logits, labels)
            - logits: raw model output, shape (batch, 41)
            - labels: ground truth, shape (batch, 41)

    Returns:
        dict of metric names and values
    """
    logits, labels = eval_pred

    # Step 1: Apply sigmoid to convert logits → probabilities
    # Sigmoid treats each label independently (unlike softmax)
    probabilities = 1 / (1 + np.exp(-logits))  # sigmoid

    # Step 2: Apply threshold to convert probabilities → binary predictions
    predictions = (probabilities >= THRESHOLD).astype(int)

    # Step 3: Compute multi-label metrics

    # Subset accuracy — strictest: entire 41-label vector must match exactly
    subset_acc = accuracy_score(labels, predictions)

    # Micro metrics — aggregate TP/FP/FN across all labels (favors common labels)
    micro_precision = precision_score(labels, predictions, average="micro", zero_division=0)
    micro_recall = recall_score(labels, predictions, average="micro", zero_division=0)
    micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)

    # Macro metrics — average per label (treats all labels equally)
    macro_precision = precision_score(labels, predictions, average="macro", zero_division=0)
    macro_recall = recall_score(labels, predictions, average="macro", zero_division=0)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)

    # Hamming loss — fraction of incorrectly predicted labels
    h_loss = hamming_loss(labels, predictions)

    return {
        "subset_accuracy": subset_acc,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "hamming_loss": h_loss,
    }


print("compute_metrics() defined.")
print("  Activation:  sigmoid")
print(f"  Threshold:   {THRESHOLD}")
print("  Metrics:     subset_accuracy, micro/macro P/R/F1, hamming_loss")

compute_metrics() defined.
  Activation:  sigmoid
  Threshold:   0.3
  Metrics:     subset_accuracy, micro/macro P/R/F1, hamming_loss


---
## Section 10 — Training Arguments

These configure HOW the model trains: batch size, epochs, learning rate, etc.

Key choices:
- **2 epochs**: Legal-BERT is pre-trained, so it needs only light fine-tuning
- **batch_size=8**: conservative for GPU memory; increase if GPU allows
- **fp16=True**: mixed precision for faster training on CUDA GPUs
- **load_best_model_at_end**: keeps the best checkpoint based on micro F1

In [48]:
# ==========================================
# TRAINING ARGUMENTS
# ==========================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=USE_FP16,
    seed=RANDOM_SEED,
    report_to="none",  # disable wandb/tensorboard for simplicity
    remove_unused_columns=False,
)

print("TrainingArguments configured.")
print(f"  Epochs:           {training_args.num_train_epochs}")
print(f"  Train batch size: {training_args.per_device_train_batch_size}")
print(f"  Eval batch size:  {training_args.per_device_eval_batch_size}")
print(f"  FP16:             {training_args.fp16}")
print(f"  Eval strategy:    {training_args.eval_strategy}")
print(f"  Best model:       {training_args.metric_for_best_model}")
print(f"  Output dir:       {training_args.output_dir}")

TrainingArguments configured.
  Epochs:           2
  Train batch size: 8
  Eval batch size:  16
  FP16:             True
  Eval strategy:    IntervalStrategy.EPOCH
  Best model:       micro_f1
  Output dir:       /kaggle/working/legal_bert_multilabel


---
## Section 11 — Trainer Setup

The HuggingFace `Trainer` handles the entire training loop:
- batching data
- forward pass
- loss computation (BCEWithLogitsLoss, automatic)
- backward pass
- optimizer step
- evaluation
- checkpointing

In [49]:
# ==========================================
# CREATE TRAINER
# ==========================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_clean,
    eval_dataset=test_dataset_clean,
    compute_metrics=compute_metrics,
)

print("Trainer created successfully.")
print(f"  Train samples: {len(train_dataset_clean)}")
print(f"  Eval samples:  {len(test_dataset_clean)}")
print(f"  Model:         {MODEL_NAME}")
print(f"  Labels:        {NUM_LABELS}")

Trainer created successfully.
  Train samples: 16073
  Eval samples:  4031
  Model:         nlpaueb/legal-bert-base-uncased
  Labels:        41


---
## Section 12 — 🚀 Training

**Execute the cell below to start training.**

Training time depends on hardware:
- **GPU (e.g., RTX 3060)**: ~10-20 minutes for 2 epochs
- **CPU**: several hours (not recommended)

The Trainer will:
1. Train for 2 epochs
2. Evaluate after each epoch
3. Save checkpoints
4. Load the best model at the end

The output won't appear because it's been run in the kaggle notebooks.

In [ ]:
# ==========================================
# START TRAINING
# ==========================================

print("Starting multi-label Legal-BERT training...")
print(f"  Device: {device}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Train samples: {len(train_dataset_clean)}")
print(f"  Eval samples:  {len(test_dataset_clean)}")
print()

train_result = trainer.train()

# Print training summary
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Total steps:     {train_result.global_step}")
print(f"  Training loss:   {train_result.training_loss:.4f}")
print(f"  Training time:   {train_result.metrics.get('train_runtime', 0):.1f} seconds")


---
## Section 13 — Evaluation

After training completes, run evaluation on the test set.
This uses the best model checkpoint loaded automatically.

In [29]:
# ==========================================
# EVALUATE ON TEST SET
# ==========================================

print("Evaluating on test set...")
eval_results = trainer.evaluate()

print("\nEVALUATION RESULTS")
print("=" * 60)
for key, value in sorted(eval_results.items()):
    if key.startswith("eval_"):
        metric_name = key.replace("eval_", "")
        if isinstance(value, float):
            print(f"  {metric_name:<25s}: {value:.4f}")
        else:
            print(f"  {metric_name:<25s}: {value}")


Evaluating on test set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



EVALUATION RESULTS
  hamming_loss             : 0.0057
  loss                     : 0.0398
  macro_f1                 : 0.2668
  macro_precision          : 0.2838
  macro_recall             : 0.2778
  micro_f1                 : 0.5963
  micro_precision          : 0.6971
  micro_recall             : 0.5210
  runtime                  : 33.9753
  samples_per_second       : 118.6450
  steps_per_second         : 3.7090
  subset_accuracy          : 0.8214


---
## Section 14 — Per-Label Analysis

After evaluation, generate detailed per-label metrics to identify
which clause categories the model handles best/worst.

In [30]:
# ==========================================
# PER-LABEL PERFORMANCE ANALYSIS
# ==========================================

# Get predictions on test set
print("Generating predictions on test set...")
predictions_output = trainer.predict(test_dataset_clean)
logits = predictions_output.predictions
true_labels = predictions_output.label_ids

# Apply sigmoid + threshold
probabilities = 1 / (1 + np.exp(-logits))
pred_labels = (probabilities >= THRESHOLD).astype(int)

# Per-label metrics
per_label_precision = precision_score(true_labels, pred_labels, average=None, zero_division=0)
per_label_recall = recall_score(true_labels, pred_labels, average=None, zero_division=0)
per_label_f1 = f1_score(true_labels, pred_labels, average=None, zero_division=0)

# Build results table
per_label_results = []
for i, label in enumerate(label_columns):
    support = int(true_labels[:, i].sum())
    predicted = int(pred_labels[:, i].sum())
    per_label_results.append({
        "label": label,
        "precision": round(per_label_precision[i], 4),
        "recall": round(per_label_recall[i], 4),
        "f1": round(per_label_f1[i], 4),
        "support": support,
        "predicted": predicted,
    })

per_label_df = pd.DataFrame(per_label_results).sort_values("f1", ascending=False)

print("\nPER-LABEL PERFORMANCE (sorted by F1)")
print("=" * 90)
print(f"  {'Label':<45s} | {'Prec':>6s} | {'Rec':>6s} | {'F1':>6s} | {'Sup':>5s} | {'Pred':>5s}")
print("  " + "-" * 85)
for _, row in per_label_df.iterrows():
    print(f"  {row['label']:<45s} | {row['precision']:>6.4f} | {row['recall']:>6.4f} | {row['f1']:>6.4f} | {row['support']:>5d} | {row['predicted']:>5d}")

# Summary
print(f"\n  Top 5 easiest labels (highest F1):")
for _, row in per_label_df.head(5).iterrows():
    print(f"    {row['label']:<40s} F1={row['f1']:.4f}")
print(f"\n  Top 5 hardest labels (lowest F1):")
for _, row in per_label_df.tail(5).iloc[::-1].iterrows():
    print(f"    {row['label']:<40s} F1={row['f1']:.4f}")


Generating predictions on test set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



PER-LABEL PERFORMANCE (sorted by F1)
  Label                                         |   Prec |    Rec |     F1 |   Sup |  Pred
  -------------------------------------------------------------------------------------
  Governing Law                                 | 0.9884 | 0.9341 | 0.9605 |    91 |    86
  Insurance                                     | 0.9143 | 1.0000 | 0.9552 |    32 |    35
  Anti-Assignment                               | 0.9211 | 0.9333 | 0.9272 |    75 |    76
  Audit Rights                                  | 0.8679 | 0.9787 | 0.9200 |    47 |    53
  Cap On Liability                              | 0.8493 | 0.9688 | 0.9051 |    64 |    73
  Expiration Date                               | 0.7778 | 0.9506 | 0.8556 |    81 |    99
  Revenue/Profit Sharing                        | 0.8462 | 0.8148 | 0.8302 |    27 |    26
  License Grant                                 | 0.7121 | 0.9038 | 0.7966 |    52 |    66
  Termination For Convenience                   | 0.805

---
## Section 15 — Model Comparison: SVM Baseline vs Legal-BERT

Compare the multi-label SVM baseline against Legal-BERT results.

In [31]:
# ==========================================
# MODEL COMPARISON: SVM Baseline vs Legal-BERT
# ==========================================

# Load SVM baseline metrics
svm_metrics_path = os.path.join(BASE_DIR, "models","charishmaganta", "models", "other", "default", "1","models", "multi_label_svm_classifier", "metrics.json")
with open(svm_metrics_path, "r") as f:
    svm_metrics = json.load(f)

print("MODEL COMPARISON: SVM Baseline vs Legal-BERT")
print("=" * 70)
print(f"  {'Metric':<25s} | {'SVM Baseline':>15s} | {'Legal-BERT':>15s} | {'Improvement':>12s}")
print("  " + "-" * 72)

comparisons = [
    ("Subset Accuracy", svm_metrics["overall"]["subset_accuracy"], eval_results["eval_subset_accuracy"]),
    ("Hamming Loss", svm_metrics["overall"]["hamming_loss"], eval_results["eval_hamming_loss"]),
    ("Micro Precision", svm_metrics["micro"]["precision"], eval_results["eval_micro_precision"]),
    ("Micro Recall", svm_metrics["micro"]["recall"], eval_results["eval_micro_recall"]),
    ("Micro F1", svm_metrics["micro"]["f1"], eval_results["eval_micro_f1"]),
    ("Macro Precision", svm_metrics["macro"]["precision"], eval_results["eval_macro_precision"]),
    ("Macro Recall", svm_metrics["macro"]["recall"], eval_results["eval_macro_recall"]),
    ("Macro F1", svm_metrics["macro"]["f1"], eval_results["eval_macro_f1"]),
]

for name, svm_val, bert_val in comparisons:
    if name == "Hamming Loss":
        improvement = svm_val - bert_val  # lower is better
    else:
        improvement = bert_val - svm_val  # higher is better
    arrow = "\u2191" if improvement > 0 else "\u2193" if improvement < 0 else "="
    print(f"  {name:<25s} | {svm_val:>15.4f} | {bert_val:>15.4f} | {arrow} {abs(improvement):>9.4f}")


MODEL COMPARISON: SVM Baseline vs Legal-BERT
  Metric                    |    SVM Baseline |      Legal-BERT |  Improvement
  ------------------------------------------------------------------------
  Subset Accuracy           |          0.7549 |          0.8214 | ↑    0.0665
  Hamming Loss              |          0.0072 |          0.0057 | ↑    0.0015
  Micro Precision           |          0.5930 |          0.6971 | ↑    0.1041
  Micro Recall              |          0.3418 |          0.5210 | ↑    0.1792
  Micro F1                  |          0.4337 |          0.5963 | ↑    0.1626
  Macro Precision           |          0.4853 |          0.2838 | ↓    0.2015
  Macro Recall              |          0.2280 |          0.2778 | ↑    0.0498
  Macro F1                  |          0.2796 |          0.2668 | ↓    0.0128


---
## Section 16 — Save Artifacts

After training, save the model, tokenizer, metrics, and label mapping
for future inference and deployment.

In [32]:
# ==========================================
# SAVE TRAINED MODEL & TOKENIZER
# ==========================================

model_save_path = os.path.join(OUTPUT_DIR, "trained_model")
os.makedirs(model_save_path, exist_ok=True)

# Save model
trainer.save_model(model_save_path)
print(f"\u2713 Model saved: {model_save_path}")

# Save tokenizer
tokenizer.save_pretrained(model_save_path)
print(f"\u2713 Tokenizer saved: {model_save_path}")

# List saved files
print("\nSaved files:")
for fname in os.listdir(model_save_path):
    fpath = os.path.join(model_save_path, fname)
    fsize = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname}: {fsize:.2f} MB")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved: /kaggle/working/legal_bert_multilabel/trained_model
✓ Tokenizer saved: /kaggle/working/legal_bert_multilabel/trained_model

Saved files:
  training_args.bin: 0.00 MB
  tokenizer.json: 0.67 MB
  model.safetensors: 417.78 MB
  tokenizer_config.json: 0.00 MB
  config.json: 0.00 MB


In [33]:
# ==========================================
# SAVE METRICS & LABEL MAPPING
# ==========================================

# Build comprehensive metrics dict
metrics_to_save = {
    "overall": {
        "subset_accuracy": round(eval_results["eval_subset_accuracy"], 4),
        "hamming_loss": round(eval_results["eval_hamming_loss"], 4),
    },
    "micro": {
        "precision": round(eval_results["eval_micro_precision"], 4),
        "recall": round(eval_results["eval_micro_recall"], 4),
        "f1": round(eval_results["eval_micro_f1"], 4),
    },
    "macro": {
        "precision": round(eval_results["eval_macro_precision"], 4),
        "recall": round(eval_results["eval_macro_recall"], 4),
        "f1": round(eval_results["eval_macro_f1"], 4),
    },
    "per_label": {
        row["label"]: {
            "precision": row["precision"],
            "recall": row["recall"],
            "f1": row["f1"],
            "support": int(row["support"]),
        }
        for _, row in per_label_df.iterrows()
    },
    "model_config": {
        "model": MODEL_NAME,
        "num_labels": NUM_LABELS,
        "max_length": MAX_LENGTH,
        "threshold": THRESHOLD,
        "epochs": int(training_args.num_train_epochs),
        "batch_size": training_args.per_device_train_batch_size,
        "fp16": training_args.fp16,
        "problem_type": "multi_label_classification",
        "loss": "BCEWithLogitsLoss",
        "activation": "sigmoid",
    },
}

metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_to_save, f, indent=2, ensure_ascii=False)
print(f"\u2713 Metrics saved: {metrics_path}")

# Save label mapping alongside model
label_map_save = os.path.join(OUTPUT_DIR, "label_mapping.json")
with open(label_map_save, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)
print(f"\u2713 Label mapping saved: {label_map_save}")


✓ Metrics saved: /kaggle/working/legal_bert_multilabel/metrics.json
✓ Label mapping saved: /kaggle/working/legal_bert_multilabel/label_mapping.json


---
## Section 17 — Final Summary

After training and evaluation are complete, this section provides
the final benchmark summary.

In [34]:
# ==========================================
# FINAL SUMMARY
# ==========================================

print("=" * 60)
print("MULTI-LABEL LEGAL-BERT CLASSIFIER \u2014 FINAL SUMMARY")
print("=" * 60)
print(f"\n  Model:           {MODEL_NAME}")
print(f"  Labels:          {NUM_LABELS}")
print(f"  Max length:      {MAX_LENGTH}")
print(f"  Threshold:       {THRESHOLD}")
print(f"  Train samples:   {len(train_dataset_clean)}")
print(f"  Test samples:    {len(test_dataset_clean)}")
print(f"  Device:          {device}")
print(f"  FP16:            {USE_FP16}")

print(f"\n  RESULTS:")
print(f"    Subset Accuracy:  {eval_results['eval_subset_accuracy']:.4f}")
print(f"    Hamming Loss:     {eval_results['eval_hamming_loss']:.4f}")
print(f"    Micro F1:         {eval_results['eval_micro_f1']:.4f}")
print(f"    Macro F1:         {eval_results['eval_macro_f1']:.4f}")
print(f"    Micro Recall:     {eval_results['eval_micro_recall']:.4f}")
print(f"    Macro Recall:     {eval_results['eval_macro_recall']:.4f}")

print(f"\n  ARTIFACTS SAVED:")
print(f"    Model:      {os.path.join(OUTPUT_DIR, 'trained_model')}")
print(f"    Metrics:    {os.path.join(OUTPUT_DIR, 'metrics.json')}")
print(f"    Labels:     {os.path.join(OUTPUT_DIR, 'label_mapping.json')}")

print(f"\n  \u2713 Multi-label Legal-BERT training COMPLETE")
print(f"  \u2713 All artifacts saved")
print(f"  \u2713 Ready for threshold calibration and inference")
print("=" * 60)


MULTI-LABEL LEGAL-BERT CLASSIFIER — FINAL SUMMARY

  Model:           nlpaueb/legal-bert-base-uncased
  Labels:          41
  Max length:      256
  Threshold:       0.3
  Train samples:   16073
  Test samples:    4031
  Device:          cuda
  FP16:            True

  RESULTS:
    Subset Accuracy:  0.8214
    Hamming Loss:     0.0057
    Micro F1:         0.5963
    Macro F1:         0.2668
    Micro Recall:     0.5210
    Macro Recall:     0.2778

  ARTIFACTS SAVED:
    Model:      /kaggle/working/legal_bert_multilabel/trained_model
    Metrics:    /kaggle/working/legal_bert_multilabel/metrics.json
    Labels:     /kaggle/working/legal_bert_multilabel/label_mapping.json

  ✓ Multi-label Legal-BERT training COMPLETE
  ✓ All artifacts saved
  ✓ Ready for threshold calibration and inference
